In [4]:
from pathlib import Path
import pandas as pd

In [148]:
store_dir = Path(r"C:\Users\user\Desktop\div\data\exports\store")
df_orders = pd.read_csv(
	store_dir / 'orders.csv',
	parse_dates=['orderdate']
	)
df_customers = pd.read_csv(
	store_dir / 'customers.csv'
	)
df_orders.head(3)

,orderid,orderdate,customerid,netamount,tax,totalamount
0,1,2004-01-27,7888,313.24,25.84,339.08
1,2,2004-01-01,4858,54.90,4.53,59.43
2,3,2004-01-17,15399,160.10,13.21,173.31


In [149]:
df_customers.head(3)

,customerid,firstname,lastname,address1,address2,city,state,zip,country,region,email,phone,creditcardtype,creditcard,creditcardexpiration,username,password,age,income,gender
0,1,VKUUXF,ITHOMQJNYX,4608499546 Dell Way,NaN,QSDPAGD,SD,24101,US,1,ITHOMQJNYX@dell.com,4608499546,1,1979279217775911,2012/03,user1,password,55,100000,M
1,2,HQNMZH,UNUKXHJVXB,5119315633 Dell Way,NaN,YNCERXJ,AZ,11802,US,1,UNUKXHJVXB@dell.com,5119315633,1,3144519586581737,2012/11,user2,password,80,40000,M
2,3,JTNRNB,LYYSHTQJRE,6297761196 Dell Way,NaN,LWVIFXJ,OH,96082,US,1,LYYSHTQJRE@dell.com,6297761196,4,8728086929768325,2010/12,user3,password,47,100000,M


In [150]:
set(df_orders.columns).intersection(set(df_customers.columns))

{'customerid'}

In [151]:
import datetime

df = pd.DataFrame(data={
	'ad' : ['ali', 'hesen', 'huseyn'],
	'dogum_tarixi' : [datetime.datetime(1999,11,20), datetime.datetime(2005,4,13), datetime.datetime(2004,2,25)],
})
df['ad'] = df['ad'].str.capitalize()
df['month'] = df['dogum_tarixi'].dt.month
df

,ad,dogum_tarixi,month
0,Ali,1999-11-20,11
1,Hesen,2005-04-13,4
2,Huseyn,2004-02-25,2


In [153]:
df_order_customer = pd.merge(df_orders, df_customers[['customerid', 'country', 'state', 'gender']], on='customerid', how='left')
# mask_aprel = df_order_customer['orderdate'].dt.month == 4
# mask_interval = (df_order_customer['orderdate'] >= '2004-04-01') & (df_order_customer['orderdate'] < '2004-09-01')
mask_interval = df_order_customer['orderdate'].between('2004-04-01', '2004-09-01', inclusive='left')
df_order_customer = df_order_customer.loc[mask_interval, :]
df_order_customer['month'] = df_order_customer['orderdate'].dt.month
df_order_customer.groupby('month')['totalamount'].sum()

month
4    216642.01
5    213395.19
6    216412.59
7    206707.38
8    210115.92
Name: totalamount, dtype: float64

In [110]:
df_order_customer.loc[df_order_customer['country'].isna(), :]

,orderid,orderdate,customerid,netamount,tax,totalamount,country,state,month


In [154]:
(
	df_order_customer
	.groupby('country')['totalamount']
	.sum()
	.sort_values(ascending=False)
	.head(3)
)

country
US           524900.19
Canada        58986.44
Australia     58782.51
Name: totalamount, dtype: float64

In [161]:
datetime.date(1999,1,2) + datetime.timedelta(days=100)

datetime.date(1999, 4, 12)

In [164]:
df_order_customer['final_date']  = df_order_customer['orderdate'] + pd.Timedelta(days=7)
df_order_customer

,orderid,orderdate,customerid,netamount,tax,totalamount,country,state,gender,month,final_date
3000,3001,2004-04-30,12411,289.62,23.89,313.51,Australia,NaN,M,4,2004-05-07
3001,3002,2004-04-08,11537,154.16,12.72,166.88,Australia,NaN,F,4,2004-04-15
3002,3003,2004-04-14,7620,293.76,24.24,318.00,US,MA,M,4,2004-04-21
3003,3004,2004-04-16,4492,95.39,7.87,103.26,US,NE,M,4,2004-04-23
3004,3005,2004-04-14,12752,247.97,20.46,268.43,Canada,NaN,M,4,2004-04-21
...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,2004-08-12,7266,397.84,32.82,430.66,US,KS,M,8,2004-08-19
7996,7997,2004-08-02,2735,283.89,23.42,307.31,US,DC,M,8,2004-08-09
7997,7998,2004-08-24,15401,280.79,23.17,303.96,UK,NaN,M,8,2004-08-31
7998,7999,2004-08-24,2993,360.98,29.78,390.76,US,UT,F,8,2004-08-31


In [176]:
df = df_order_customer.set_index('orderdate').sort_index()
df.loc['2004-04'].head(5)

,orderid,customerid,netamount,tax,totalamount,country,state,gender,month,final_date
orderdate,,,,,,,,,,
2004-04-01,3318,948,378.00,31.19,409.19,US,HI,M,4,2004-04-08
2004-04-01,3107,11467,82.32,6.79,89.11,UK,NaN,F,4,2004-04-08
2004-04-01,3574,14240,197.81,16.32,214.13,Canada,NaN,M,4,2004-04-08
2004-04-01,3575,14614,304.81,25.15,329.96,South Africa,NaN,F,4,2004-04-08
2004-04-01,3907,2716,382.53,31.56,414.09,US,TN,F,4,2004-04-08


In [187]:
df_gunluk = df.resample('D')['totalamount'].sum()
# df_gunluk - df_gunluk.shift(1)
df_gunluk.diff(1)

orderdate
2004-04-01        NaN
2004-04-02    1395.63
2004-04-03    2099.12
2004-04-04   -1172.10
2004-04-05   -3092.28
               ...   
2004-08-27    1672.55
2004-08-28     407.80
2004-08-29   -1462.45
2004-08-30   -1962.35
2004-08-31    3044.77
Freq: D, Name: totalamount, Length: 153, dtype: float64

In [186]:
df_gunluk.shift(3)

orderdate
2004-04-01        NaN
2004-04-02        NaN
2004-04-03        NaN
2004-04-04    5952.88
2004-04-05    7348.51
               ...   
2004-08-27    8971.06
2004-08-28    5559.26
2004-08-29    5961.00
2004-08-30    7633.55
2004-08-31    8041.35
Freq: D, Name: totalamount, Length: 153, dtype: float64

In [143]:
df_orders = pd.read_csv(
	store_dir / 'orders.csv',
	)
df_customers = pd.read_csv(
	store_dir / 'customers.csv'
	)
df_customers.head(2)

,customerid,firstname,lastname,address1,address2,city,state,zip,country,region,email,phone,creditcardtype,creditcard,creditcardexpiration,username,password,age,income,gender
0,1,VKUUXF,ITHOMQJNYX,4608499546 Dell Way,NaN,QSDPAGD,SD,24101,US,1,ITHOMQJNYX@dell.com,4608499546,1,1979279217775911,2012/03,user1,password,55,100000,M
1,2,HQNMZH,UNUKXHJVXB,5119315633 Dell Way,NaN,YNCERXJ,AZ,11802,US,1,UNUKXHJVXB@dell.com,5119315633,1,3144519586581737,2012/11,user2,password,80,40000,M


In [142]:
# df_orders['orderdate'].str.split('-').str[1].astype('int')
pd.to_numeric(df_orders['orderdate'].str.split('-').str[1])

0         1
1         1
2         1
3         1
4         1
         ..
11995    12
11996    12
11997    12
11998    12
11999    12
Name: orderdate, Length: 12000, dtype: int64

In [123]:
def yalanci_month(setir): # '2004-01-27'
	return int(setir.split('-')[1])

df_orders['orderdate'].apply(yalanci_month)

0         1
1         1
2         1
3         1
4         1
         ..
11995    12
11996    12
11997    12
11998    12
11999    12
Name: orderdate, Length: 12000, dtype: int64

1